# Mini-Bengio: generalization to unseen word combinations

**Classroom demonstration (15–25 minutes, CPU only)** — inspired by Bengio, Ducharme, Vincent & Jauvin, *A Neural Probabilistic Language Model* (JMLR, 2003).

**Research question.** Can a neural next-word predictor assign high probability to an *unseen combination of known words*? Compare (A) a count-based trigram with bigram back-off and (B) a neural trigram with learned embeddings, one `tanh` hidden layer, and a softmax distribution.

**How to run:** Open this `.ipynb` in Google Colab or Jupyter; select **Runtime → Run all**. It uses `torch`, `numpy` and `matplotlib` (preinstalled on Colab); **no GPU, API key or external data**. A `.py` version is also provided.

This is an **illustration**, not a faithful reproduction of the original paper's performance comparisons. The corpus is deliberately artificial, the baseline is simplistic, and the task is very easy for a neural model.


In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(1)
print('PyTorch version:', torch.__version__)


PyTorch version: 2.10.0+cpu


## 1. Prepare the training/test split
The entire language has just 24 possible triples. We deliberately remove six **subject–relation combinations**, while ensuring every individual word is observed during training.

In [2]:
# Eight subjects, three relations, six possible target words = 17 words total.
animals = ["cat", "dog", "fox", "wolf"]
vehicles = ["car", "bus", "van", "train"]
relations = ["needs", "has", "moves"]
endings = {
    "animal":  {"needs": "food", "has": "fur", "moves": "walks"},
    "vehicle": {"needs": "fuel", "has": "wheels", "moves": "rolls"},
}

# Hold out one relation per chosen subject. Every WORD remains in training.
held_out_contexts = {
    ("dog", "needs"), ("wolf", "has"), ("fox", "moves"),
    ("bus", "needs"), ("train", "has"), ("van", "moves"),
}

all_triples = [
    (s, r, endings["animal" if s in animals else "vehicle"][r])
    for s in animals + vehicles for r in relations
]
train = [row for row in all_triples if row[:2] not in held_out_contexts]
test  = [row for row in all_triples if row[:2] in held_out_contexts]

vocab = animals + vehicles + relations + list(endings["animal"].values()) + list(endings["vehicle"].values())
word_id = {w: i for i, w in enumerate(vocab)}
assert len(vocab) == 17 and len(train) == 18 and len(test) == 6
assert set(w for triple in test for w in triple).issubset(set(w for triple in train for w in triple))

print(f'Vocabulary: {len(vocab)} words | Train: {len(train)} triples | Test: {len(test)} unseen triples')
print('TRAIN:')
for s, r, w in train: print(f'  {s:<6} {r:<6} {w}')
print('HELD OUT:')
for s, r, w in test: print(f'  {s:<6} {r:<6} {w}')


Vocabulary: 17 words | Train: 18 triples | Test: 6 unseen triples
TRAIN:
  cat    needs  food
  cat    has    fur
  cat    moves  walks
  dog    has    fur
  dog    moves  walks
  fox    needs  food
  fox    has    fur
  wolf   needs  food
  wolf   moves  walks
  car    needs  fuel
  car    has    wheels
  car    moves  rolls
  bus    has    wheels
  bus    moves  rolls
  van    needs  fuel
  van    has    wheels
  train  needs  fuel
  train  moves  rolls
HELD OUT:
  dog    needs  food
  fox    moves  walks
  wolf   has    fur
  bus    needs  fuel
  van    moves  rolls
  train  has    wheels


## 2. Count-based baseline
The trigram model looks up the exact two-word context. For the six unseen contexts it backs off to a one-word context (the relation), where the correct ending competes with another ending.

In [3]:
trigram_counts = Counter(train)
bigram_counts = Counter((r, w) for _, r, w in train)

def count_probability(subject, relation, target):
    # Full trigram context (subject, relation) if observed; otherwise back off
    # to the shorter context (relation) and use relative frequencies.
    total_3 = sum(n for (s, r, _), n in trigram_counts.items()
                  if s == subject and r == relation)
    if total_3:
        return trigram_counts[(subject, relation, target)] / total_3
    total_2 = sum(n for (r, _), n in bigram_counts.items() if r == relation)
    return bigram_counts[(relation, target)] / total_2

print('Count-model probability of the correct word in unseen contexts:')
for s, r, w in test:
    print(f'  P({w:6} | {s:5} {r:5}) = {count_probability(s, r, w):.2f}')


Count-model probability of the correct word in unseen contexts:
  P(food   | dog   needs) = 0.50
  P(walks  | fox   moves) = 0.50
  P(fur    | wolf  has  ) = 0.50
  P(fuel   | bus   needs) = 0.50
  P(rolls  | van   moves) = 0.50
  P(wheels | train has  ) = 0.50
